# E-commerce: agregaciones masivas con Spark por capas
**Dataset:** Ecommerce Behavior Data (Millones de interacciones)
**Tecnología:** Apache Spark (Modo Local) sobre HDFS

En esta práctica vamos a procesar varios millones de registros reales aplicando la **Arquitectura Medallón**.
El dataset se compone de **múltiples archivos CSV** a la vez.

**¡ATENCIÓN! - Posible Error 403 Forbidden de Kaggle**
> ⚠️ Si al ejecutar la celda de Kaggle te da un`HTTPError: 403 Client Error: Forbidden`, significa que tu archivo`kaggle.json` no tiene credenciales válidas, o que has superado el límite de descargas de la API. ¡Asegúrate de haber generado un nuevo token en Kaggle.com!

In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, desc, when , trim

In [1]:

spark = SparkSession.builder \
    .appName("Practica_Ventas_Kaggle") \
    .master("local[*]") \
    .getOrCreate()

# Ajusta particiones para local (muy importante)
# En local, el default puede ser malo (demasiadas o muy pocas tareas).
# Si tienes 8 núcleos (ejemplo)
# spark = (
#     SparkSession.builder
#     .appName("Intro_Spark")
#     .master("local[*]")
#     .config("spark.sql.shuffle.partitions", "16")  # prueba 8, 16, 32
#     .getOrCreate()
# )
# Regla práctica
# dataset grande + local → prueba 8 / 16 / 32
# - si pones demasiado alto → overhead
# - si pones demasiado bajo → poco paralelismo

# Ejemplo: muestra la versión de PySpark
try:
    print(f"Versión de PySpark {spark.version}.")
except NameError:
    print("SparkSession no inicializada; ejecuta la celda de creación de spark.")

### 🥉 Bronce: Ingesta Masiva
Este dataset es gigante (varios GB) y contiene archivos CSV que representan diferentes meses (Octubre, Noviembre...).
HDFS brillará aquí guardándolos distribuidos, y Spark los leerá todos a la vez.

In [2]:
!pip install -q opendatasets
import opendatasets as od

# Dataset masivo de comportamiento en Ecommerce
dataset_url = "https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store"

od.download(dataset_url)



Skipping, found downloaded files in "./ecommerce-behavior-data-from-multi-category-store" (use force=True to force download)


In [ ]:
# Ingestamos TODOS los CSV (*.csv) a una única ruta en Bronce.
!hdfs dfs -mkdir -p /data/bronze/ecommerce/
!hdfs dfs -put -f ecommerce-behavior-data-from-multi-category-store/*.csv /data/bronze/ecommerce/

### 🥈 Plata: Estandarización
Uniremos todos los CSVs y los validaremos (quitando basuras/nulos) para convertirlos a Parquet.

In [22]:
ruta_bronze = "hdfs://namenode:9000/data/bronze/ecommerce/*.csv"

# 1. Leer un directorio entero de CSV con millones de filas haciendo que adivine el esquema
df_ventas = spark.read.option("header", "true").option("inferSchema", "true").csv(ruta_bronze)

total = df_ventas.count()
print(f"Total de registros: {total:,}")

Total de registros: 109,950,743


In [21]:
# VER SI ALGUN ID TIENE NULOS
# Ver una solo columna
print("user_id nulos:",df_ventas.filter(col("user_id").isNull()).count())
# Ver varias a la vez: 
id_cols = ["product_id", "category_id", "user_id", "user_session"]

df_ventas.select([
    count(when(col(c).isNull(), c)).alias(f"{c}_nulls")
    for c in id_cols
]).show()

# Extra: detectar nulos o vacíos (strings)
df_ventas.filter(
    col("user_session").isNull() | (trim(col("user_session")) == "")
).show(20, truncate=False)

user_id nulos: 0
+----------------+-----------------+-------------+------------------+
|product_id_nulls|category_id_nulls|user_id_nulls|user_session_nulls|
+----------------+-----------------+-------------+------------------+
|               0|                0|            0|                12|
+----------------+-----------------+-------------+------------------+

+-------------------+----------+----------+-------------------+---------------------------+-------+-------+---------+------------+
|event_time         |event_type|product_id|category_id        |category_code              |brand  |price  |user_id  |user_session|
+-------------------+----------+----------+-------------------+---------------------------+-------+-------+---------+------------+
|2019-11-09 15:32:27|cart      |19700004  |2053013559104766575|NULL                       |kabrita|37.77  |539704497|NULL        |
|2019-11-09 17:15:24|cart      |1005083   |2053013555631882655|electronics.smartphone     |honor  |566.27 |5

In [5]:
df_ventas.show(5)

+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code| brand| price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|2019-11-01 00:00:00|      view|   1003461|2053013555631882655|electronics.smart...|xiaomi|489.07|520088904|4d3b30da-a5e4-49d...|
|2019-11-01 00:00:00|      view|   5000088|2053013566100866035|appliances.sewing...|janome|293.65|530496790|8e5f4f83-366c-4f7...|
|2019-11-01 00:00:01|      view|  17302664|2053013553853497655|                NULL| creed| 28.31|561587266|755422e7-9040-477...|
|2019-11-01 00:00:01|      view|   3601530|2053013563810775923|appliances.kitche...|    lg|712.87|518085591|3bfb58cd-7892-48c...|
|2019-11-01 00:00:01|      view|   1004775|2053013555631882655|electronics.smart...|xiaomi

In [6]:
# 2. Transformación principal: Filtra las interacciones de compras defectuosas (precios nulos) y eventos que no sean compras
df_plata = df_ventas.na.drop(subset=["price", "brand"]).filter(col("event_type") == "purchase")
df_plata.show(5)


+-------------------+----------+----------+-------------------+--------------------+---------+------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|    brand| price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+---------+------+---------+--------------------+
|2019-11-01 00:01:04|  purchase|   1005161|2053013555631882655|electronics.smart...|   xiaomi|211.92|513351129|e6b7ce9b-1938-4e2...|
|2019-11-01 00:04:51|  purchase|   1004856|2053013555631882655|electronics.smart...|  samsung|128.42|562958505|0f039697-fedc-40f...|
|2019-11-01 00:05:34|  purchase|  26401669|2053013563651392361|                NULL|  lucente|109.66|541854711|c41c44d5-ef9b-41b...|
|2019-11-01 00:06:33|  purchase|   1801881|2053013554415534427|electronics.video.tv|  samsung| 488.8|557746614|4d76d6d3-fff5-488...|
|2019-11-01 00:06:34|  purchase|   5800823|2053013553945772349|electr

In [7]:
# 3. Escribir a Plata
ruta_silver = "hdfs://namenode:9000/data/silver/ecommerce/"
df_plata.write.mode("overwrite").parquet(ruta_silver)
print("Escritura a plata realizada")

Escritura a plata realizada


### 🥇 Oro: Transformaciones de Negocio (KPIs)

In [25]:
# 1. Leemos de nuestra Plataforma Silver
df_silver = spark.read.parquet(ruta_silver)

# 2. Regla de Negocio: ¿Qué marca (brand) ha facturado más dinero sumando todos los meses?
df_top_ventas = df_silver.groupBy("brand").agg(sum("price").alias("facturacion_total")).orderBy(desc("facturacion_total"))

# 3. Servir a BI
ruta_gold = "hdfs://namenode:9000/data/gold/top_ventas_ecommerce/"
df_top_ventas.write.mode("overwrite").parquet(ruta_gold)
df_top_ventas.show(10)

+----------------+-----------------+-------------+------------------+
|product_id_nulls|category_id_nulls|user_id_nulls|user_session_nulls|
+----------------+-----------------+-------------+------------------+
|               0|                0|            0|                 0|
+----------------+-----------------+-------------+------------------+

+-------+--------------------+
|  brand|   facturacion_total|
+-------+--------------------+
|  apple|  2.38721793699998E8|
|samsung|1.0127741348000328E8|
| xiaomi|2.0453899250000067E7|
| huawei|   9664104.089999977|
|     lg|   8626906.719999993|
|   acer|   6924026.050000006|
|lucente|   6651658.940000006|
|   sony|   6341082.979999995|
|   oppo|   5901500.520000032|
| lenovo|   4450744.830000005|
+-------+--------------------+
only showing top 10 rows



### 1) Usa un formato columnar (Parquet) en vez de CSV

Si estás leyendo CSV, ese suele ser el mayor cuello de botella.

- **CSV** = lento (parseo, tipos, texto)
    
- **Parquet** = mucho más rápido (columnar, comprimido, schema)

In [26]:
# Ejemplo filtro de nulos sobre df_silver
id_cols = ["product_id", "category_id", "user_id", "user_session"]

df_silver.select([
    count(when(col(c).isNull(), c)).alias(f"{c}_nulls")
    for c in id_cols
]).show()


+----------------+-----------------+-------------+------------------+
|product_id_nulls|category_id_nulls|user_id_nulls|user_session_nulls|
+----------------+-----------------+-------------+------------------+
|               0|                0|            0|                 0|
+----------------+-----------------+-------------+------------------+



### 2) Define esquema manualmente (no`inferSchema=True`)

`inferSchema` en archivos grandes hace una pasada extra y puede ser lento.

Mejor:

In [28]:
from pyspark.sql.types import *

schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", LongType(), True),
    StructField("category_id", LongType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_session", StringType(), True),
])

ruta_bronze = "hdfs://namenode:9000/data/bronze/ecommerce/*.csv"

# 1. Leer un directorio entero de CSV con millones de filas haciendo que adivine el esquema
# df_ventas = spark.read.option("header", "true").option("inferSchema", "true").csv(ruta_bronze)

df = spark.read.csv(ruta_bronze, header=True, schema=schema)
df.show(10)

+--------------------+----------+----------+-------------------+--------------------+--------+------+---------+--------------------+
|          event_time|event_type|product_id|        category_id|       category_code|   brand| price|  user_id|        user_session|
+--------------------+----------+----------+-------------------+--------------------+--------+------+---------+--------------------+
|2019-11-01 00:00:...|      view|   1003461|2053013555631882655|electronics.smart...|  xiaomi|489.07|520088904|4d3b30da-a5e4-49d...|
|2019-11-01 00:00:...|      view|   5000088|2053013566100866035|appliances.sewing...|  janome|293.65|530496790|8e5f4f83-366c-4f7...|
|2019-11-01 00:00:...|      view|  17302664|2053013553853497655|                NULL|   creed| 28.31|561587266|755422e7-9040-477...|
|2019-11-01 00:00:...|      view|   3601530|2053013563810775923|appliances.kitche...|      lg|712.87|518085591|3bfb58cd-7892-48c...|
|2019-11-01 00:00:...|      view|   1004775|2053013555631882655|elect

In [30]:
# Muestra cuántas particiones tiene actualmente el DataFrame.
# Esto te ayuda a entender cómo Spark dividirá el trabajo en tareas paralelas.
print(df.rdd.getNumPartitions())

# Reorganiza el DataFrame para que tenga exactamente 16 particiones.
# Spark redistribuye los datos (shuffle), lo que puede mejorar el paralelismo
# si antes había muy pocas particiones o estaban mal distribuidas.
df = df.repartition(16)

110


## Explicación

###`print(df.rdd.getNumPartitions())`

- Devuelve el **número actual de particiones** del DataFrame.
    
- En Spark, las particiones son “trozos” de datos que se procesan en paralelo.
    
- Si tienes muy pocas, no aprovechas los cores.
    
- Si tienes demasiadas, hay overhead (muchas tareas pequeñas).
    

###`df = df.repartition(16)`

- Fuerza al DataFrame a tener **16 particiones**.
    
- Esto puede mejorar rendimiento en operaciones posteriores (`groupBy`,`join`, etc.) si 16 encaja mejor con tu máquina.
    
- **Ojo:**`repartition()` suele implicar **shuffle** (mueve datos entre particiones), así que no conviene usarlo sin necesidad.
    

---

## Cuándo usarlo

- ✅ Antes de operaciones pesadas si ves mala distribución / pocas particiones.
    
- ✅ Si quieres ajustar el paralelismo en local (por ejemplo 8, 16, 32).
    
- ❌ No lo uses en cada paso del pipeline, porque el shuffle cuesta tiempo.

## Avanzado

In [31]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel

# ==============================
# 1) Crear sesión de Spark (local)
# ==============================
spark = (
    SparkSession.builder
    .appName("Analisis_Local_Spark")
    .master("local[*]")  # Usa todos los núcleos disponibles de tu PC
    .config("spark.driver.memory", "8g")  # Memoria para Spark (ajusta según tu RAM)
    .config("spark.sql.shuffle.partitions", "16")  
    # Número de particiones por defecto en operaciones pesadas (groupBy, join, orderBy)
    .config("spark.sql.adaptive.enabled", "true")  
    # Spark puede ajustar particiones automáticamente en algunos casos (AQE)
    .getOrCreate()
)

# ==============================
# 2) Leer datos (ejemplo)
# ==============================
df = spark.read.parquet(ruta_silver) 
# Mejor usar Parquet (más rápido que CSV)

# ==============================
# 3) Ver cuántas particiones tiene el DataFrame
# ==============================
print("Particiones actuales:", df.rdd.getNumPartitions())
# Esto te dice en cuántos "trozos" Spark dividirá el trabajo en paralelo

# ==============================
# 4) Ajustar particiones si hace falta
# ==============================
df = df.repartition(16)
# Reparte los datos en 16 particiones
# Útil si quieres mejorar paralelismo para operaciones pesadas
# Ojo: repartition hace shuffle (mueve datos), así que no lo uses muchas veces sin necesidad

print("Particiones después de repartition:", df.rdd.getNumPartitions())

# ==============================
# 5) Cachear solo si lo vas a reutilizar varias veces
# ==============================
df.persist(StorageLevel.MEMORY_AND_DISK)
# Guarda el DataFrame en memoria (y en disco si no cabe)
# Bueno si vas a hacer varios análisis sobre el mismo df

df.count()
# Acción para "materializar" el cache (sin esto, todavía no se guarda realmente)

# ==============================
# 6) Ejemplo de análisis
# ==============================
df.groupBy("event_type").count().show()
# groupBy suele hacer shuffle; por eso ayuda tener bien configuradas las particiones

Particiones actuales: 20
Particiones después de repartition: 16
+----------+-------+
|event_type|  count|
+----------+-------+
|  purchase|1528301|
+----------+-------+



## Explicación fácil (qué hace cada cosa)

###`local[*]`

Usa todos los núcleos de tu computador.
➡️ Más paralelo = mejor uso de tu CPU.

###`spark.driver.memory = 8g`

Le da memoria a Spark.
➡️ Si tienes poca memoria, Spark va lento o puede fallar.

###`spark.sql.shuffle.partitions = 16`

Controla cuántas particiones usa Spark en operaciones pesadas (`groupBy`,`join`, etc.).
➡️ En local,`16` suele ir mejor que`200` (que muchas veces es demasiado).

###`df.rdd.getNumPartitions()`

Te dice cuántas particiones tiene el DataFrame.
➡️ Sirve para entender si estás usando pocas o demasiadas.

###`df.repartition(16)`

Reorganiza los datos en 16 particiones.
➡️ Ayuda a repartir mejor el trabajo, pero cuesta (shuffle).

###`persist(MEMORY_AND_DISK)`

Guarda el DataFrame para reutilizarlo varias veces.
➡️ Evita recalcular todo desde cero en cada operación.

---

## Regla fácil para recordar

- **Explorar rápido** → usa muestra (`sample`)

- **Operaciones pesadas** → revisa particiones

- **Repetir análisis** → usa`cache/persist`

- **Velocidad de lectura** → usa **Parquet**

In [34]:
# Cerrar la sesión de Spark al finalizar
try:
    spark.stop()
    print("SparkSession detenida.")
except Exception as e:
    print("No se pudo detener SparkSession:", e)

SparkSession detenida.
